In [2]:
import pandas as pd
import numpy as np

Load the datasets

In [3]:
df = pd.read_excel("/content/online_retail_II.xlsx",sheet_name="Year 2009-2010")
df2 = pd.read_excel("/content/online_retail_II.xlsx",sheet_name="Year 2010-2011")

Combine the two datasets using concat

In [4]:
df_new = pd.concat([df,df2])

Give the information of datasets

In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 525461 entries, 0 to 525460
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   Invoice      525461 non-null  object        
 1   StockCode    525461 non-null  object        
 2   Description  522533 non-null  object        
 3   Quantity     525461 non-null  int64         
 4   InvoiceDate  525461 non-null  datetime64[ns]
 5   Price        525461 non-null  float64       
 6   Customer ID  417534 non-null  float64       
 7   Country      525461 non-null  object        
dtypes: datetime64[ns](1), float64(2), int64(1), object(4)
memory usage: 32.1+ MB


check the missing values or null values from the datasets

In [7]:
df_new.isnull().sum()

,0
Invoice,0
StockCode,0
Description,4382
Quantity,0
InvoiceDate,0
Price,0
Customer ID,243007
Country,0


#description and Customer Id id having null values
 we have to remove the null description rows we cannot analyze or recommend a product if you do not know what it is.

#keep the null customer Id rows beacuse
 Market Basket Analysis groups transactions by the Invoice (Order ID), not the Customer ID.
 A transaction can still have a valid invoice number and a list of products even if the customer checked out as a guest without an account.
 Dropping these would delete 243,007 valid product rows that are useful for your analysis.

drop the null description rows

In [8]:
df_new = df_new.dropna(subset=['Description'])

In [9]:
df_new.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1062989 entries, 0 to 541909
Data columns (total 8 columns):
 #   Column       Non-Null Count    Dtype         
---  ------       --------------    -----         
 0   Invoice      1062989 non-null  object        
 1   StockCode    1062989 non-null  object        
 2   Description  1062989 non-null  object        
 3   Quantity     1062989 non-null  int64         
 4   InvoiceDate  1062989 non-null  datetime64[ns]
 5   Price        1062989 non-null  float64       
 6   Customer ID  824364 non-null   float64       
 7   Country      1062989 non-null  object        
dtypes: datetime64[ns](1), float64(2), int64(1), object(4)
memory usage: 73.0+ MB


In [10]:
df_new.isnull().sum()

,0
Invoice,0
StockCode,0
Description,0
Quantity,0
InvoiceDate,0
Price,0
Customer ID,238625
Country,0


Make copy of df_new

In [11]:
df_final= df_new.copy()

Drop the cancelled transaction ids  which is marked as 'C'with invoive number

In [12]:
# Convert Invoice to string to allow text filtering

df_final['Invoice'] = df_final['Invoice'].astype(str)

# Exclude rows where Invoice starts with 'C'

df_clean = df_final[~df_final['Invoice'].str.startswith('C')]

# Strip whitespace from descriptions to avoid duplicates like "Milk " and "Milk"

df_clean['Description'] = df_clean['Description'].str.strip()


/tmp/ipykernel_5379/3722426506.py:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_clean['Description'] = df_clean['Description'].str.strip()


In [13]:
df_clean.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1043495 entries, 0 to 541909
Data columns (total 8 columns):
 #   Column       Non-Null Count    Dtype         
---  ------       --------------    -----         
 0   Invoice      1043495 non-null  object        
 1   StockCode    1043495 non-null  object        
 2   Description  1043491 non-null  object        
 3   Quantity     1043495 non-null  int64         
 4   InvoiceDate  1043495 non-null  datetime64[ns]
 5   Price        1043495 non-null  float64       
 6   Customer ID  805620 non-null   float64       
 7   Country      1043495 non-null  object        
dtypes: datetime64[ns](1), float64(2), int64(1), object(4)
memory usage: 71.7+ MB


Total number of unique transactions (N) for Support calculation

In [16]:
# Total number of unique transactions (N) for Support calculation
total_transactions = df_clean["Invoice"].nunique()
print("Total number of unique transactions (N):", total_transactions)

Total number of unique transactions (N): 41061


create the product basket list
 1.Group by Invoice and create a list of items for each transaction

In [21]:
Basket = df_clean.groupby('Invoice')['Description'].apply(list).tolist()

Filter out basket that only contain 1 product

In [24]:
Basket =[basket for basket in Basket if len(basket)>1]

print(f"Total valid Baskets for pair analysis: {len(Basket)}")

Total valid Baskets for pair analysis: 36552


 Generate Product Pairs and Calculate Frequencies

 import itertools and Counter to find the most frequent combinations of products.

In [25]:
import itertools
from collections import Counter

# Generate and count unique item pairs per transaction

pair_counter = Counter()

for basket in Basket:
    # Use set to avoid counting duplicate items within the same order
    unique_items = sorted(list(set(basket)))

    # Generate all 2-item combinations
    pairs = itertools.combinations(unique_items, 2)
    pair_counter.update(pairs)

# Convert to DataFrame to see the top results

pairs_df = pd.DataFrame(pair_counter.most_common(10), columns=['Product Pair', 'Frequency'])
print(pairs_df)


                                        Product Pair  Frequency
0  (RED HANGING HEART T-LIGHT HOLDER, WHITE HANGI...       1240
1  (WOODEN FRAME ANTIQUE WHITE, WOODEN PICTURE FR...       1114
2  (JUMBO BAG RED RETROSPOT, JUMBO STORAGE BAG SUKI)       1096
3  (STRAWBERRY CERAMIC TRINKET BOX, SWEETHEART CE...       1052
4  (GREEN REGENCY TEACUP AND SAUCER, ROSES REGENC...       1024
5  (JUMBO BAG RED RETROSPOT, JUMBO SHOPPER VINTAG...       1008
6     (HEART OF WICKER LARGE, HEART OF WICKER SMALL)       1005
7  (JUMBO BAG PINK POLKADOT, JUMBO BAG RED RETROS...        966
8  (60 TEATIME FAIRY CAKE CASES, PACK OF 60 PINK ...        926
9    (JUMBO BAG RED RETROSPOT, JUMBO BAG STRAWBERRY)        926


Calculate the advanced metrics (Support, Confidence, and Lift) for these pairs.
1.Support - How popular the item combination is.
2.Confidence - How sure you are about the second item.
3.Lift (Real Connection vs. Luck) -
  Lift = 1: The items are together by pure random coincidence.
  Lift > 1: The items have a strong, genuine bond.
  The goal: highest Lift values for your final recommendations.

In [26]:
# Count how many total transactions each unique item appears in

item_counts = df_clean.groupby('Description')['Invoice'].nunique().to_dict()
item_support = {item: count / total_transactions for item, count in item_counts.items()}

Compute Association Metric


In [27]:
metrics_list = []

for pair, pair_count in pair_counter.items():
    item_A, item_B = pair

    # 1. Support: percentage of all transactions containing both items

    support = pair_count / total_transactions

    # 2. Confidence: how often B is bought when A is purchased

    confidence_A_to_B = pair_count / item_counts[item_A]
    confidence_B_to_A = pair_count / item_counts[item_B]

    # 3. Lift: how much more likely they are bought together than by random chance

    lift = support / (item_support[item_A] * item_support[item_B])

    metrics_list.append({
        'Item A': item_A,
        'Item B': item_B,
        'Pair Count': pair_count,
        'Support': round(support, 4),
        'Confidence (A->B)': round(confidence_A_to_B, 4),
        'Confidence (B->A)': round(confidence_B_to_A, 4),
        'Lift': round(lift, 4)
    })

Display Top Recommentation by Lift

In [28]:
metrics_df = pd.DataFrame(metrics_list)

# Sort by Lift descending to find the strongest rules

final_analysis_df = metrics_df.sort_values(by='Lift', ascending=False)
print(final_analysis_df.head(10))

                                     Item A  \
3346963         NUMBER TILE VINTAGE FONT, 3   
3346935         NUMBER TILE VINTAGE FONT, 2   
3346936         NUMBER TILE VINTAGE FONT, 2   
3346937         NUMBER TILE VINTAGE FONT, 2   
568363     F.FAIRY S/3 SML CANDLE, LAVENDER   
2455878     TRAY FOR TV DINNER , DOLLY GIRL   
568378   F.FAIRY S/3 SML CANDLE,LILY/VALLEY   
568361     F.FAIRY S/3 SML CANDLE, LAVENDER   
3346886         NUMBER TILE VINTAGE FONT, 0   
3346938         NUMBER TILE VINTAGE FONT, 2   

                                      Item B  Pair Count  Support  \
3346963          NUMBER TILE VINTAGE FONT, 7           1      0.0   
3346935          NUMBER TILE VINTAGE FONT, 3           1      0.0   
3346936          NUMBER TILE VINTAGE FONT, 4           1      0.0   
3346937          NUMBER TILE VINTAGE FONT, 5           1      0.0   
568363   F.FAIRY,CANDLE IN GLASS,LILY/VALLEY           1      0.0   
2455878         TRAY FOR TV DINNER ,SPACEBOY           1      0.0   